In [4]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from loguru import logger
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

In [5]:
import tomllib

configfile = Path("../config.toml").resolve()
with configfile.open("rb") as f:
    config = tomllib.load(f)
config

{'raw': 'data/raw',
 'preprocessed': 'data/preprocessed',
 'cleaned': 'data/cleaned',
 'feature_added': 'data/feature_added',
 'processed': 'data/processed',
 'input': '_chat.txt',
 'current': 'whatsapp_current.parq',
 'preprocess_csv': 'whatsapp_preprocess.csv',
 'preprocess_parq': 'whatsapp_preprocess.parq',
 'cleaned_csv': 'whatsapp_cleaned.csv',
 'cleaned_parq': 'whatsapp_cleaned.parq',
 'feature_engineered_csv': 'whatsapp_feature.csv',
 'feature_engineered_parq': 'whatsapp_feature.parq',
 'time_series_plot_png': 'time_series_plot.png',
 'categories_plot_png': 'categories_plot.png',
 'distribution_plot_png': 'distribution_plot.png',
 'correlation_plot_png': 'correlation_plot.png',
 'dimensionality_plot_png': 'dimensionality_plot.png',
 'datetime_format': '%d-%m-%Y %H:%M',
 'drop_authors': []}

In [6]:
import os

root = Path("..").resolve()
# Feature-engineered files live in feature_added with timestamped names: whatsapp-<datetime>-features.csv/.parq
feature_added_dir = root / Path(config["feature_added"])

# Find latest parquet or CSV matching the pipeline's naming pattern
parq_files = sorted(feature_added_dir.glob("whatsapp-*-features.parq"), key=os.path.getmtime, reverse=True)
csv_files = sorted(feature_added_dir.glob("whatsapp-*-features.csv"), key=os.path.getmtime, reverse=True)

if parq_files:
    datafile = parq_files[0]
    logger.info(f"Using latest parquet: {datafile.name}")
elif csv_files:
    datafile = csv_files[0]
    logger.info(f"Using latest CSV: {datafile.name}")
else:
    # Fallback: fixed name from config (if pipeline ever writes a single file)
    datafile = feature_added_dir / config.get("feature_engineered_parq", config.get("feature_engineered_csv", "whatsapp_feature.parq"))

if not datafile.exists():
    logger.warning(
        f"No feature-engineered file found in {feature_added_dir}. "
        "First run: preprocess → clean → feature engineering (e.g. python -m dav_bas_hv.main), and check the timestamp!"
    )

2026-03-16 15:58:20.335 | INFO     | __main__:<module>:13 - Using latest parquet: whatsapp-20260316-142043-features.parq


In [7]:
# Load the datafile resolved in the previous cell (parquet or CSV)
if datafile.exists():
    if datafile.suffix == ".parq":
        df = pd.read_parquet(datafile)
    else:
        df = pd.read_csv(datafile, parse_dates=["timestamp"])
    logger.info(f"Loaded {len(df)} rows from {datafile.name}")
else:
    df = None  # run the pipeline first

2026-03-16 15:58:20.402 | INFO     | __main__:<module>:7 - Loaded 10136 rows from whatsapp-20260316-142043-features.parq


In [9]:
# Meeting-up questions: is_question == 1 and mentions_meet_up == 1
df["is_meeting_up_question"] = (
    (df["is_question"] == 1) & (df["mentions_meet_up"] == 1)
).astype(int)

# Group by living_in_city only (no year dimension)
total_messages = df.groupby("living_in_city").size()
question_counts = df.groupby("living_in_city")["is_question"].sum()
meeting_up_counts = df.groupby("living_in_city")["is_meeting_up_question"].sum()
total_meeting_up = meeting_up_counts.sum()
total_questions = question_counts.sum()

# Share of all meeting-up questions per group (what we had before)
pct_of_meeting_up = (meeting_up_counts / total_meeting_up * 100).round(2)

# Ratio: meeting-up questions as % of total messages per group
pct_meeting_up_of_messages = (meeting_up_counts / total_messages * 100).round(2)

stats = pd.DataFrame({
    "group": ["non_city_living", "city_living"],
    "living_in_city": [0, 1],
    "total_messages": [int(total_messages.get(0, 0)), int(total_messages.get(1, 0))],
    "total_questions": [int(question_counts.get(0, 0)), int(question_counts.get(1, 0))],
    "total_meeting_up_questions": [int(meeting_up_counts.get(0, 0)), int(meeting_up_counts.get(1, 0))],
    "pct_of_all_meeting_up_questions": [pct_of_meeting_up.get(0, 0), pct_of_meeting_up.get(1, 0)],
    "pct_meeting_up_of_own_messages": [pct_meeting_up_of_messages.get(0, 0), pct_meeting_up_of_messages.get(1, 0)],
})
print("Per-group stats (total messages, meeting-up questions, and ratio):")
print(stats)
print(f"\nTotal questions: {int(total_questions)}")
print(f"\nGrand total meeting-up questions: {int(total_meeting_up)}")

Per-group stats (total messages, meeting-up questions, and ratio):
             group  living_in_city  total_messages  total_questions  \
0  non_city_living               0            4114              598   
1      city_living               1            6022              738   

   total_meeting_up_questions  pct_of_all_meeting_up_questions  \
0                         114                             42.7   
1                         153                             57.3   

   pct_meeting_up_of_own_messages  
0                            2.77  
1                            2.54  

Total questions: 1336

Grand total meeting-up questions: 267


### Feedback on original graph used in img/final

#### Original:
-  Gestalt / Preattentive processing: door timeseries te gebruiken vestig je de aandacht op variatie in tijd. Je had beter een barplot kunnen gebruiken, als je boodschap gaat over % meeting up questions per groep.

- Story: je hebt eigenlijk geen verhaal. “This graph shows the opposite”, maar wat is dan je verhaal?

- Spurious Correlation: Verder besteed je nergens echt aandacht aan het feit dat je conclusies veroorzaakt zouden kunnen worden door een persoon die toevallig afwijkend gedrag geeft en toevallig een feature bezit.

#### Gemini:
- Categorical vs time-series
    - *Issue*: Chart implies time series, data implies categorical
    - *Solution*: Use grouped bar chart
- Gestalt principles:
    - *Issue*: Too much focus on continuity
    - *Solution*:
        - Similarity: Use distinct colors for both groups
        - Proximity: Placing bars next to each other for comparison
- Preattentive processing:
    - *Issue*: Currently, preattentive feature is the slope, which is misleading for graph
    - *Solution*: Focus on difference in height/length to make other group pop and show that it is more
- Spurious correlation:
    - *Issue*: Small N gives hint that this might also be a distribution prolem
    - *Solution*: Show the spread using a distribution for instance
